In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('CT_Train_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

# # 新しいボリュームのサイズ (各軸を半分にする)
# new_shape = tuple([dim // 2 for dim in x_train.shape[1:]])

# # サイズを半分に縮小
# x_train_resized = np.zeros((x_train.shape[0], *new_shape))  # 新しい形に合わせて初期化

# for i in range(x_train.shape[0]):
#     # 各画像を縮小
#     x_train_resized[i] = scipy.ndimage.zoom(x_train[i], (0.5, 0.5, 0.5), order=3)
# print('Resized train vol_shape:', x_train_resized.shape[1:])
# print('Resized train shape:', x_train_resized.shape)
# x_train = x_train_resized

In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    return mse

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]

In [9]:
model3D_3 = vxm.networks.VxmDense1((128, 256, 256), nb_features, int_steps=0)
model3D_3.to(device)
optimizer = optim.Adam(model3D_3.parameters(), lr=1e-4)

5
6
7
6
7
6
7
6
7
6
7
64
128
8
9
64
10
8
9
64
10
8
9
64
10
8
9
64
10
8
9
64
10
11
11
11
[64, 128, 128]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [10]:
# NotdecoderHight
from tqdm.notebook import tqdm
transformer = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

# 3D ガウシアンフィルタを適用する関数
def gaussian_smooth_3d(tensor, kernel_size=5, sigma=1.0):
    """ 3D ガウシアンフィルタで displacement field をスムージング """
    # 3D Gaussian Kernel の作成
    from scipy.ndimage import gaussian_filter
    tensor_np = tensor.cpu().numpy()
    smoothed_np = gaussian_filter(tensor_np, sigma=[0, 0, sigma, sigma, sigma])  # チャネル方向にはフィルタ適用しない
    return torch.tensor(smoothed_np, dtype=torch.float32, device=tensor.device)


# エポック数と最小ロスの設定
epochs = 40000
# epochs = 1000
best_loss = float('inf')
shift_range = 1

# ロスや他のメトリクスを記録するリスト
losses = []
loss_vecs = []
loss_images = []
loss_hightVecs = []
for epoch in tqdm(range(epochs)):
    # 100エポックごとにshift_rangeを増やす
    if epoch % 500 == 0 and epoch > 0:
    # if epoch % 100 == 0 and epoch > 0:
        shift_range += 5
        print(f"Epoch {epoch}: Increasing shift range to ±{shift_range} pixels.")

    # 学習データのバッチを取得
    train_batch, _ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32).to(device)

    # 画像サイズを設定
    B, D, H, W = 4, 8, 16, 16  # バッチサイズと画像の次元

    # displacement_field をボクセルごとにランダムに作成
    displacement_field = (torch.rand((B, 3, D, H, W), dtype=torch.float32) * 2 - 1) * shift_range
    displacement_field = displacement_field.to(device)

    # 3D Gaussian Smoothing を適用
    displacement_field = gaussian_smooth_3d(displacement_field, sigma=2.0)
    displacement_field = torch.nn.functional.interpolate(displacement_field, size=(128,256,256), mode='trilinear', align_corners=False)

    # 位置をずらした画像を生成
    moving_images2 = transformer(moving_images, displacement_field)

    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, Vec = model3D_3(moving_images, moving_images2)

    # 損失を計算
    loss_vec = MSE_Loss(displacement_field, Vec) * 0.1
    loss_image = MSE_Loss(moving_images2, transformed_image) * 100
    
    loss = loss_vec + loss_image

    # 逆伝播
    loss.backward()
    optimizer.step()

    # モデルを保存
    torch.save(model3D_3.state_dict(), 'model_VXM_3D_duwl_omomi_NodecoderHH_256kaiii.pth')

    # エポックごとのロスを保存
    losses.append(loss.cpu().item())
    loss_vecs.append(loss_vec.cpu().item())
    loss_images.append(loss_image.cpu().item())
    # loss_hightVecs.append(loss_hightVec.cpu().item())
   
    # エポックごとのロスの表示
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, loss_vec: {loss_vec:.4f}, loss_image: {loss_image:.4f}, Shift Range: ±{shift_range} pixels")

In [ ]:
# NotdecoderHight
import pywt

from tqdm.notebook import tqdm
transformer = vxm.layers.SpatialTransformer((128, 256, 256))

# 3D ガウシアンフィルタを適用する関数
def gaussian_smooth_3d(tensor, kernel_size=5, sigma=1.0):
    """ 3D ガウシアンフィルタで displacement field をスムージング """
    # 3D Gaussian Kernel の作成
    from scipy.ndimage import gaussian_filter
    tensor_np = tensor.cpu().numpy()
    smoothed_np = gaussian_filter(tensor_np, sigma=[0, 0, sigma, sigma, sigma])  # チャネル方向にはフィルタ適用しない
    return torch.tensor(smoothed_np, dtype=torch.float32, device=tensor.device)

def wavelet_decompose(tensor, wavelet='haar', level=1):
    """
    tensor: (B, 1, D, H, W)
    出力: LL, HH
    """
    LL_list, HH_list = [], []
    for i in range(tensor.size(0)):
        x = tensor[i, 0].cpu().numpy()  # (D, H, W)
        coeffs = pywt.wavedecn(x, wavelet=wavelet, mode='symmetric', level=level)
        LL = coeffs[0]
        # 高周波成分は全キーから平均を取る（もしくは特定キーを選ぶ）
        HH = sum(torch.tensor(v) for v in coeffs[1].values()) / len(coeffs[1])
        LL_list.append(torch.tensor(LL).unsqueeze(0))
        HH_list.append(HH.unsqueeze(0))
    LL = torch.stack(LL_list).unsqueeze(1).to(tensor.device)  # (B, 1, D', H', W')
    HH = torch.stack(HH_list).unsqueeze(1).to(tensor.device)
    return LL, HH
    
# エポック数と最小ロスの設定
epochs = 40000
# epochs = 1000
best_loss = float('inf')
shift_range = 1

# ロスや他のメトリクスを記録するリスト
losses = []
loss_vecs = []
loss_images = []
loss_hightVecs = []
for epoch in tqdm(range(epochs)):
    # 100エポックごとにshift_rangeを増やす
    if epoch % 500 == 0 and epoch > 0:
    # if epoch % 100 == 0 and epoch > 0:
        shift_range += 5
        print(f"Epoch {epoch}: Increasing shift range to ±{shift_range} pixels.")

    # 学習データのバッチを取得
    train_batch, _ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)

    # 画像サイズを設定
    B, D, H, W = 2, 8, 16, 16  # バッチサイズと画像の次元

    # displacement_field をボクセルごとにランダムに作成
    displacement_field = (torch.rand((B, 3, D, H, W), dtype=torch.float32) * 2 - 1) * shift_range
    displacement_field = displacement_field

    # 3D Gaussian Smoothing を適用
    displacement_field = gaussian_smooth_3d(displacement_field, sigma=2.0)
    displacement_field = torch.nn.functional.interpolate(displacement_field, size=(128,256,256), mode='trilinear', align_corners=False)

    # 位置をずらした画像を生成
    moving_images2 = transformer(moving_images, displacement_field)

    moving_images_reshaped = moving_images.unsqueeze(1)
    moving_images2_reshaped = moving_images2.unsqueeze(1)
    
    LL1, HH1 = wavelet_decompose(moving_images_reshaped)
    LL2, HH2 = wavelet_decompose(moving_images2_reshaped)
    
    LL1 = LL1.squeeze(2).squeeze(2).to(device)  # or LL1 = LL1.view(B, C, D, H, W)
    LL2 = LL2.squeeze(2).squeeze(2).to(device)
    HH1 = HH1.squeeze(2).squeeze(2).to(device)
    HH2 = HH2.squeeze(2).squeeze(2).to(device)
    displacement_field = displacement_field.to(device)  # もし損失計算に使うならこれも
    moving_images = moving_images.to(device)  # もし損失計算に使うならこれも
    moving_images2 = moving_images2.to(device)  # もし損失計算に使うならこれも

    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, Vec = model3D_3(LL1, LL2, HH1, HH2, moving_images)

    # 損失を計算
    loss_vec = MSE_Loss(displacement_field, Vec) * 0.1
    loss_image = MSE_Loss(moving_images2, transformed_image) * 100
    
    loss = loss_vec + loss_image

    # 逆伝播
    loss.backward()
    optimizer.step()

    # モデルを保存
    torch.save(model3D_3.state_dict(), 'model_VXM_3D_duwl_omomi_NodecoderHH_flita4.pth')

    # エポックごとのロスを保存
    losses.append(loss.cpu().item())
    loss_vecs.append(loss_vec.cpu().item())
    loss_images.append(loss_image.cpu().item())
    # loss_hightVecs.append(loss_hightVec.cpu().item())
   
    # エポックごとのロスの表示
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, loss_vec: {loss_vec:.4f}, loss_image: {loss_image:.4f}, Shift Range: ±{shift_range} pixels")

  0%|          | 0/40000 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Temp\ipykernel_4496\910048877.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
C:\Users\user\anaconda3\envs\nn\lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 1 is too high: all coefficients will experience boundary effects.
  warnings.warn(


After remaining conv: torch.Size([2, 32, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
torch.Size([2, 3, 128, 256, 256])
Epoch 1/40000, Loss: 0.0009, loss_vec: 0.0002, loss_image: 0.0007, Shift Range: ±1 pixels
After remaining conv: torch.Size([2, 32, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
torch.Size([2, 3, 128, 256, 256])
Epoch 2/40000, Loss: 0.0010, loss_vec: 0.0002, loss_image: 0.0009, Shift Range: ±1 pixels
After remaining conv: torch.Size([2, 32, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
torch.Size([2, 3, 128, 256, 256])
Epoch 3/40000, Loss: 0.0008, loss_vec: 0.0002, loss_image: 0.0007, Shift Range: ±1 pixels
After remaining conv: torch.Size([2, 32, 128, 256, 256])
After remaining conv: torch.Size([2, 16, 128, 256, 256])
A

In [3]:
import torch
import torch.nn.functional as F
from pytorch_wavelets import DWT3D
from tqdm.notebook import tqdm
import vxm  # SpatialTransformer が必要

# GPU設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3D Gaussian smoothing を PyTorch で実装
import torch.nn as nn
import torch.nn.functional as F

class GaussianSmoothing3D(nn.Module):
    def __init__(self, channels, kernel_size=5, sigma=2):
        super().__init__()
        self.channels = channels
        grid = torch.arange(kernel_size) - kernel_size // 2
        gauss = torch.exp(-0.5 * (grid / sigma).pow(2))
        kernel_1d = gauss / gauss.sum()
        kernel_3d = kernel_1d[:, None, None] * kernel_1d[None, :, None] * kernel_1d[None, None, :]
        kernel = kernel_3d.expand(channels, 1, kernel_size, kernel_size, kernel_size)
        self.register_buffer('weight', kernel)
        self.groups = channels

    def forward(self, x):
        return F.conv3d(x, weight=self.weight, groups=self.groups, padding='same')

# Wavelet分解（GPU対応）
def wavelet_decompose(tensor, wavelet='haar'):
    dwt = DWT3D(J=1, wave=wavelet).to(tensor.device)
    LL, HH = dwt(tensor)
    return LL, HH[0]  # HHはリスト（J=1）なので[0]

# 設定
transformer = vxm.layers.SpatialTransformer((128, 256, 256))
gaussian_smoother = GaussianSmoothing3D(channels=3).to(device)
epochs = 40000
best_loss = float('inf')
shift_range = 1

losses, loss_vecs, loss_images = [], [], []

for epoch in tqdm(range(epochs)):
    if epoch % 500 == 0 and epoch > 0:
        shift_range += 5
        print(f"Epoch {epoch}: Increasing shift range to ±{shift_range} pixels.")

    # データロード
    train_batch, _ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32, device=device)  # (B, D, H, W)

    # パラメータ
    B, D, H, W = moving_images.shape
    moving_images = moving_images.unsqueeze(1)  # (B, 1, D, H, W)

    # displacement field (GPUで直接作成)
    displacement_field = ((torch.rand((B, 3, D, H, W), device=device) * 2 - 1) * shift_range)
    displacement_field = gaussian_smoother(displacement_field)
    displacement_field = F.interpolate(displacement_field, size=(128, 256, 256), mode='trilinear', align_corners=False)

    # 変形画像の生成
    moving_images2 = transformer(moving_images.squeeze(1), displacement_field)

    # Wavelet 分解
    LL1, HH1 = wavelet_decompose(moving_images)
    LL2, HH2 = wavelet_decompose(moving_images2.unsqueeze(1))

    # モデル入力・ロス計算
    optimizer.zero_grad()
    transformed_image, Vec = model3D_3(LL1, LL2, HH1, HH2, moving_images)

    loss_vec = MSE_Loss(displacement_field, Vec) * 0.1
    loss_image = MSE_Loss(moving_images2, transformed_image) * 100
    loss = loss_vec + loss_image

    loss.backward()
    optimizer.step()

    # ログ記録
    losses.append(loss.item())
    loss_vecs.append(loss_vec.item())
    loss_images.append(loss_image.item())

    # モデル保存は100エポックごと
    if epoch % 100 == 0:
        torch.save(model3D_3.state_dict(), f'model_VXM_3D_duwl_epoch{epoch}.pth')

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, loss_vec: {loss_vec:.4f}, loss_image: {loss_image:.4f}, Shift Range: ±{shift_range} pixels")


ImportError: cannot import name 'DWT3D' from 'pytorch_wavelets' (C:\Users\user\anaconda3\envs\nn\lib\site-packages\pytorch_wavelets\__init__.py)

In [ ]:
print("LL1.shape:", LL1.shape)
print("LL2.shape:", LL2.shape)
print("HH1.shape:", HH1.shape)
print("HH2.shape:", HH2.shape)

In [ ]:
LL1 = LL1.squeeze(2).squeeze(2)  # or LL1 = LL1.view(B, C, D, H, W)
LL2 = LL2.squeeze(2).squeeze(2)
HH1 = HH1.squeeze(2).squeeze(2)
HH2 = HH2.squeeze(2).squeeze(2)

In [2]:
!pip install pytorch_wavelets


     ---------------------------------------- 0.0/54.9 kB ? eta -:--:--
     ---------------------------------------- 54.9/54.9 kB 2.8 MB/s eta 0:00:00
